# Lab | Embeddings
## Find the answer before the customer finishes typing

You work for a digital bank. Every day, thousands of customers write to support: *my card has not arrived*, *I was charged twice*, *why is the exchange rate different?* Most of these questions have been asked, and answered, many times before.

The support team wants a tool that takes a new message and instantly shows the agent the **most similar past messages** and their category, so the agent can reuse the answer. Later, the same search will feed a chatbot (that is Day 5's RAG). You will build that tool with **sentence embeddings** and check, with numbers, whether it beats a classic TF-IDF search.

**The data** is the Banking77 dataset: 13,083 real customer messages to a bank, each labelled with one of 77 categories (called **intents**), such as `card_arrival` or `exchange_rate`. `data/banking_train.csv` has 10,003 messages, which play the role of the past messages, and `data/banking_test.csv` has 3,080 messages, which play the role of new ones.

**This lab practises** semantic search with sentence embeddings. You will use, from today's **Embeddings and Semantic Search** lesson: `SentenceTransformer.encode`, cosine similarity, unit-length vectors and the dot product, top-*k* search with `np.argsort`, the TF-IDF search, and KMeans with the adjusted Rand index.

**By the end, you can:**

1. Implement a top-*k* semantic search with sentence embeddings and cosine similarity.
2. Evaluate it against a TF-IDF baseline and explain where each method fails.

Each challenge ends with a **checkpoint** that tells you what you should see. If your numbers are very different, fix that before moving on.

## Setup

The first run downloads the `all-MiniLM-L6-v2` model (about 90 MB, internet needed once). If the import fails, install the library from a terminal with `pip install sentence-transformers`.

In [1]:
pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import logging
os.environ["HF_HUB_VERBOSITY"] = "error"      # hide Hugging Face download notices
os.environ["TOKENIZERS_PARALLELISM"] = "false"
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").setLevel(logging.ERROR)
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"   # allow torch and scikit-learn to share OpenMP
os.environ["OMP_NUM_THREADS"] = "1"           # avoid the thread clash in KMeans
...

import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from transformers.utils import logging as transformers_logging
transformers_logging.set_verbosity_error()
transformers_logging.disable_progress_bar()

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

pd.set_option("display.max_colwidth", 90)

In [3]:
train = pd.read_csv("../data/banking_train.csv")
test = pd.read_csv("../data/banking_test.csv")
print(train.shape, test.shape, train["category"].nunique())
train.head()

(10003, 2) (3080, 2) 77


,text,category
0,I am still waiting on my card?,card_arrival
1,What can I do if my card still hasn't arrived after 2 weeks?,card_arrival
2,I have been waiting over a week. Is the card still coming?,card_arrival
3,Can I track my card while it is in the process of delivery?,card_arrival
4,"How do I know if I will get my card, or if it is lost?",card_arrival


**Checkpoint:** `(10003, 2) (3080, 2) 77`, and two columns, `text` and `category`.

## Explore the data

1. How many training messages does each intent have? Look at the smallest and the largest with `value_counts()`.
2. Print 5 random training messages with their intent (`sample(5, random_state=0)`).
3. Print all the intents whose name contains `card` (`train["category"].unique()` and a list comprehension). How easy do you think it is to tell them apart?

In [4]:
# 1-3. Explore the data

# 1. Messages per intent: smallest and largest
counts = train["category"].value_counts()
print("Smallest intents:")
print(counts.tail(5))
print("\nLargest intents:")
print(counts.head(5))
print(f"\nMin {counts.min()}, max {counts.max()}, mean {counts.mean():.0f}")

# 2. Five random messages with their intent
print()
print(train.sample(5, random_state=0))

# 3. All intents whose name contains "card"
card_intents = sorted([c for c in train["category"].unique() if "card" in c])
print(f"\n{len(card_intents)} intents contain 'card':")
for c in card_intents:
    print(" ", c)


Smallest intents:
category
lost_or_stolen_card         82
card_swallowed              61
card_acceptance             59
virtual_card_not_working    41
contactless_not_working     35
Name: count, dtype: int64

Largest intents:
category
card_payment_fee_charged                            187
direct_debit_payment_not_recognised                 182
balance_not_updated_after_cheque_or_cash_deposit    181
wrong_amount_of_cash_received                       180
cash_withdrawal_charge                              177
Name: count, dtype: int64

Min 35, max 187, mean 130

                                                                                           text  \
7145                                                 How can I transfer money to a beneficiary?   
834                                                    How is my ATM deposit not available yet?   
3585  There is a vendor name I don't recognize on a payment from my account. I don't think I...   
4497   My statement shows charges f

**Question:** what did you notice about the intents with `card` in their name?

27 out of 77 intents involve `card` in their name and it seems like many differ only by a small detail. That makes them hard to distinguish even for a human.

## Challenge 1: compare sentences

Start small. Load the model and measure how similar a few pairs of customer messages are.

1. Load `all-MiniLM-L6-v2` into a variable called `model`.
2. For every pair in `pairs`, encode the two sentences and print their cosine similarity, rounded to 2 decimals, next to the pair.

In [5]:
pairs = [
    ("My card still hasn't arrived", "When will I receive my new card?"),
    ("I was charged twice for the same purchase", "Why is there a duplicate payment on my statement?"),
    ("I was charged twice for the same purchase", "How do I change my PIN?"),
    ("How do I top up my account?", "My top up failed"),
    ("My card was stolen", "My card was not stolen"),
]

# 1. Load the model
model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Similarity of each pair
for a, b in pairs:
    vectors = model.encode([a,b])
    cosine_score = cosine_similarity(vectors)[0,1]
    print(f'{cosine_score:.2f} | {a!r} / {b!r}')

0.65 | "My card still hasn't arrived" / 'When will I receive my new card?'
0.60 | 'I was charged twice for the same purchase' / 'Why is there a duplicate payment on my statement?'
0.13 | 'I was charged twice for the same purchase' / 'How do I change my PIN?'
0.63 | 'How do I top up my account?' / 'My top up failed'
0.96 | 'My card was stolen' / 'My card was not stolen'


**Checkpoint:** the duplicate-payment pair scores about 0.6 although it shares almost no words, the PIN pair scores about 0.1, and the last pair scores above 0.9.

**Question:** the last pair scores higher than any other. Is that a problem for a support tool? What would happen if a bot answered *My card was not stolen* with the answer to *My card was stolen*?

Yes, embeddings measure topic rather than meaning, so the negation barely changes the vector and a bot relying only on similarity could tell a customer whose card is fine that it has been blocked and replaced, or vice versa, which is why automatic answers need a human check or a confidence threshold.

## Challenge 2: build the index

A search engine encodes all the documents once and keeps the vectors.

1. Encode all the training messages (`train["text"].tolist()`) into `train_vectors`. Use `normalize_embeddings=True` so that every vector has length 1, and `batch_size=64`. It takes between 10 seconds and a minute, depending on your computer.
2. Print the shape of `train_vectors` and the length of the first vector (`np.linalg.norm`).

In [6]:
# 1. Encode the training messages
train_vectors = model.encode(
    train['text'].tolist(),
    normalize_embeddings=True,
    batch_size=64
)

# 2. Shape and length
print('Index shape:', train_vectors.shape)
print('Length of the first vector:', np.linalg.norm(train_vectors[0]).round(4))

Index shape: (10003, 384)
Length of the first vector: 1.0


**Checkpoint:** `(10003, 384) 1.0`.

## Challenge 3: semantic search

Complete `semantic_search(query, k=5)`. It must return a DataFrame with the `k` most similar training messages, best first, and three columns: `score`, `category` and `text`.

The steps, as in the lesson:

1. Encode the query with `normalize_embeddings=True`.
2. Compute the similarity with every training message. Because all vectors have length 1, the dot product **is** the cosine similarity: `train_vectors @ query_vector`.
3. Find the positions of the `k` highest scores with `np.argsort` (it sorts from lowest to highest).
4. Build the DataFrame from those positions.

In [7]:
def semantic_search(query, k=5):
    # 1. Encode the query
    query_vector = model.encode(query, normalize_embeddings=True)

    # 2. Similarity with every training message
    scores = train_vectors @ query_vector

    # 3. Positions of the k highest scores
    top = np.argsort(scores)[::-1][:k]

    # 4. DataFrame with score, category and text
    return pd.DataFrame({
        'score': scores[top].round(2),
        'category': train['category'].iloc[top].values,
        'text': train['text'].iloc[top].values
    })


semantic_search("I can't find my card anywhere")

,score,category,text
0,0.95,lost_or_stolen_card,I cant find my card.
1,0.92,lost_or_stolen_card,I can't find my card! Can you help?
2,0.92,lost_or_stolen_card,"I can't find my card, it is lost."
3,0.91,lost_or_stolen_card,I can't find my card! What should I do?
4,0.90,card_arrival,Where is my card?


**Checkpoint:** the first four results are lost-card messages with the intent `lost_or_stolen_card` and scores above 0.9. The fifth, *Where is my card?*, belongs to `card_arrival`.

## Challenge 4: a TF-IDF baseline

Is the embedding model worth it? Build the Day 1 alternative and compare.

1. Fit a `TfidfVectorizer()` (default settings) on the training texts and keep the training matrix in `train_tfidf`.
2. Write `tfidf_search(query, k=5)`, returning the same three columns as `semantic_search`. TF-IDF rows already have length 1, so the dot product is again the cosine similarity. The product of two sparse matrices is sparse: `.toarray().ravel()` turns the result into a flat NumPy array.
3. Run both searches on each query in `queries`.

In [8]:
queries = [
    "I can't find my card anywhere",
    "Someone took money from my account without asking",
    "Why did the exchange rate change so much?",
    "The ATM gave me less cash than I asked for",
]

# 1. Fit the vectoriser
tfidf = TfidfVectorizer()
train_tfidf = tfidf.fit_transform(train['text'])

# 2. tfidf_search
def tfidf_search(query, k=5):
    scores = (train_tfidf @ tfidf.transform([query]).T).toarray().ravel()
    top = np.argsort(scores)[::-1][:k]
    return pd.DataFrame({
        'score': scores[top].round(2),
        'category': train['category'].iloc[top].values,
        'text': train['text'].iloc[top].values
    })
    

# 3. Compare
for query in queries:
    print('Semantic Search:')
    display(semantic_search(query))

    print('TF-IDF Search:')
    display(tfidf_search(query))

Semantic Search:


,score,category,text
0,0.95,lost_or_stolen_card,I cant find my card.
1,0.92,lost_or_stolen_card,I can't find my card! Can you help?
2,0.92,lost_or_stolen_card,"I can't find my card, it is lost."
3,0.91,lost_or_stolen_card,I can't find my card! What should I do?
4,0.90,card_arrival,Where is my card?


TF-IDF Search:


,score,category,text
0,0.87,get_physical_card,I can't find my PIN anywhere.
1,0.74,card_acceptance,Can I use my card anywhere?
2,0.69,order_physical_card,Can I receive my card anywhere?
3,0.67,card_acceptance,Is there anywhere I can't use my card?
4,0.65,card_acceptance,Can I pay with my card anywhere?


Semantic Search:


...

,score,category,text
0,0.76,wrong_exchange_rate_for_cash_withdrawal,Why did my exchange rate change?
1,0.58,exchange_rate,Does the exchange rate change?
2,0.51,card_payment_wrong_exchange_rate,I want to change the wrong exchange rate.
3,0.48,exchange_rate,How's the exchange rate?
4,0.47,card_payment_wrong_exchange_rate,Why was the exchange rate so wrong when I bought something!


Semantic Search:


,score,category,text
0,0.98,wrong_amount_of_cash_received,The ATM gave me less cash than requested
1,0.93,wrong_amount_of_cash_received,I got less cash at the ATM than I asked for. Why?
2,0.91,wrong_amount_of_cash_received,Why did I get less cash than what I asked in the ATM?
3,0.91,wrong_amount_of_cash_received,The ATM did not give me as much cash as I requested.
4,0.91,wrong_amount_of_cash_received,Why did the ATM give me less money than what I requested?


TF-IDF Search:


,score,category,text
0,0.79,wrong_amount_of_cash_received,The ATM gave me less cash than requested
1,0.75,wrong_amount_of_cash_received,I got less cash at the ATM than I asked for. Why?
2,0.72,wrong_amount_of_cash_received,Why did I get less cash than what I asked in the ATM?
3,0.72,wrong_amount_of_cash_received,"I got less than what i asked for in the ATM, why?"
4,0.71,wrong_amount_of_cash_received,How can I fix this? The ATM gave me more cash than I'd asked for.


**Checkpoint:** for the first query, TF-IDF's top result is *I can't find my PIN anywhere*.

**Question:** for which queries does TF-IDF go wrong, and why? For which does it do as well as semantic search? Write one query of your own where you expect TF-IDF to fail, and check it.

**TF-IDF goes wrong on queries 1 and 2**, because it only matches shared words, not meaning:

- *I can't find my card anywhere*: the top result is *I can't find my PIN anywhere* (`get_physical_card`), and the rest are `card_acceptance` messages like *Can I use my card anywhere?*. The words *can't*, *find* and *anywhere* dominate, so TF-IDF misses that the customer has lost their card. Semantic search returns four `lost_or_stolen_card` messages with scores above 0.9.
- *Someone took money from my account without asking*: TF-IDF matches *took* and *without*, and returns *Someone took my card without my permission* (`lost_or_stolen_card`) and even *The ATM took my card*. The customer is describing an unrecognised transaction, which semantic search finds (`card_payment_not_recognised`, `cash_withdrawal_not_recognised`) although those messages use different words (*accessed*, *withdrew*, *wasn't me*).

**TF-IDF does about as well on queries 3 and 4**, because the customer uses the same words as the past messages:

- *Why did the exchange rate change so much?*: both methods return exchange-rate intents, and both have the same top result. The mix of `exchange_rate`, `card_payment_wrong_exchange_rate` and `wrong_exchange_rate_for_cash_withdrawal` comes from the intents themselves overlapping, not from the method. TF-IDF is slightly weaker: *I want to change the wrong exchange rate* only matches the word *change*.
- *The ATM gave me less cash than I asked for*: both return five `wrong_amount_of_cash_received` messages. TF-IDF's fifth result, *The ATM gave me **more** cash than I'd asked for*, has the opposite meaning but the same intent. It happens to be right here only because both directions share one intent.


In [9]:
my_query = "The cash machine kept my plastic"
display('Semantic search:',semantic_search(my_query))
display('TF-IDF search:', tfidf_search(my_query))

'Semantic search:'

,score,category,text
0,0.69,wrong_amount_of_cash_received,It kept my cash
1,0.62,wrong_amount_of_cash_received,The machine gave me the wrong amount of cash.
2,0.58,wrong_amount_of_cash_received,Where'd the rest of my cash go from the ATM
3,0.57,wrong_amount_of_cash_received,The cash I wanted wasn't there.
4,0.57,card_swallowed,My card was eaten by the cash machine what do I do?


'TF-IDF search:'

,score,category,text
0,0.76,wrong_amount_of_cash_received,It kept my cash
1,0.65,card_swallowed,The ATM kept my card?
2,0.48,card_swallowed,what do I need to do if the ATM kept my card?
3,0.46,atm_support,Can I use the card at any cash machine?
4,0.45,change_pin,Can I change my pin at a cash machine?


**My own query:** *The cash machine kept my plastic*. I expected TF-IDF to fail because the training messages say *ATM* and *card*, not *cash machine* and *plastic*. In fact, both methods got the top result wrong (*It kept my cash*, `wrong_amount_of_cash_received`), and TF-IDF did slightly better: it found two `card_swallowed` messages in places 2 and 3 probably due to the rare word *kept*. Semantic search ranked `card_swallowed` only 5th. The small model does not seem to know that *plastic* is slang for a card, and the word *cash* pulled it towards cash problems. So embeddings are not automatically better: they only know the synonyms they saw in training, and TF-IDF wins when a rare shared word carries the meaning.

## Challenge 5: measure it

Four queries are anecdotes. To compare the methods properly, treat the 3,080 test messages as new customer messages, find the most similar training message for each one, and check whether it has the same intent. The share of correct intents is the **top-1 accuracy** of the search.

1. Encode the test messages into `test_vectors` (normalised, as before).
2. `test_vectors @ train_vectors.T` gives a 3,080 × 10,003 matrix with the similarity of every test message to every training message. For each row, `.argmax(axis=1)` gives the position of the most similar training message. Use it to get the predicted intents and compute the accuracy.
3. Do the same with TF-IDF. The similarity matrix is sparse here, and its `argmax(axis=1)` returns a matrix: wrap it with `np.asarray(...).ravel()`.
4. Print both accuracies.

In [10]:
# 1. Encode the test messages
test_vectors = model.encode(
    test['text'].tolist(),
    normalize_embeddings=True,
    batch_size=64
)

# 2. Embedding accuracy
similarity = test_vectors @ train_vectors.T
best = similarity.argmax(axis=1)
pred_embed = train['category'].to_numpy()[best]
acc_embed = (pred_embed == test['category'].to_numpy()).mean()

# 3. TF-IDF accuracy
test_tfidf = tfidf.transform(test['text'])
similarity_tfidf = test_tfidf @ train_tfidf.T
best_tfidf = np.asarray(similarity_tfidf.argmax(axis=1)).ravel()
pred_tfidf = train['category'].to_numpy()[best_tfidf]
acc_tfidf = (pred_tfidf == test['category'].to_numpy()).mean()

# 4. Print both
print(f'Top accuracy, embeddings: {acc_embed:.3f}')
print(f'Top accuracy, TF-IDF: {acc_tfidf:.3f}')

Top accuracy, embeddings: 0.920
Top accuracy, TF-IDF: 0.791


**Checkpoint:** about 0.92 for embeddings and 0.79 for TF-IDF.

5. Now look at the mistakes of the embedding search. Build a DataFrame with the test text, the true intent and the predicted intent, keep the rows where they differ, and print the 8 most frequent (true, predicted) pairs with `value_counts()`. Then print a few of the mistaken messages.

In [11]:
# 5. Mistakes of the embedding search
results = pd.DataFrame({
    'text': test['text'],
    'true': test['category'],
    'predicted': pred_embed,
})
mistakes = results[results['true'] != results['predicted']]
print(f'{len(mistakes)} mistakes out of {len(results)}\n')

print(mistakes[['true', 'predicted']].value_counts().head(8))
print()
mistakes.sample(8, random_state=0)

245 mistakes out of 3080

true                            predicted             
top_up_failed                   top_up_reverted           6
pin_blocked                     change_pin                5
card_arrival                    card_delivery_estimate    4
fiat_currency_support           exchange_via_app          4
card_delivery_estimate          card_arrival              4
exchange_via_app                fiat_currency_support     4
top_up_by_bank_transfer_charge  transfer_fee_charged      4
card_payment_not_recognised     compromised_card          4
Name: count, dtype: int64



,text,true,predicted
1101,I think my statement has a fraudulent charge,card_payment_not_recognised,transaction_charged_twice
2939,"I will need to get a new card soon, how do I order?",card_about_to_expire,order_physical_card
879,I transferred some money but the recipient is having trouble withdrawing.,transfer_not_received_by_recipient,declined_cash_withdrawal
649,I do not see a top-up processed.,pending_top_up,top_up_failed
2295,Why hasn't my transfer been made?,failed_transfer,transfer_not_received_by_recipient
2341,What methods can I use to add money to my account?,transfer_into_account,top_up_by_cash_or_cheque
1631,"My card is still pending, I've been waiting a while.",pending_card_payment,card_arrival
1626,"I was double charged, and the second charge is showing as ""pending"". How long will it ...",pending_card_payment,transaction_charged_twice


**Question:** look at the most frequent confusions. Are they really mistakes? What does this tell you about the data and about the 0.92?

They aren't mistakes, in fact both the true & predicted category can apply. 
This tells us the 77 intents overlap and each message gets only one label, so even a perfect human would not score 100%. The 0.92 therefore underestimates how useful the tool is. Many "wrong" matches would still help the agent, and in practice the agent sees the top 5 matches, not just the first. Accuracy against a single label is a strict measure when the categories themselves are fuzzy.

## Challenge 6: group messages without labels

Imagine the bank had no intents yet, only messages. Could you discover the groups from the embeddings? Test it on six intents, whose true labels you hide from KMeans and only use to check the result.

1. Build a boolean mask of the training messages that belong to the six intents in `six`: `mask = train["category"].isin(six).to_numpy()`. Use it to keep those messages (`train[mask]`) and their vectors (`train_vectors[mask]`).
2. Cluster the vectors with `KMeans(n_clusters=6, n_init=10, random_state=0)`, and show a `pd.crosstab` of the clusters against the true intents.
3. Compute the adjusted Rand index (ARI) of the clustering.
4. Do the same with the TF-IDF rows of those messages (`train_tfidf[mask]`), and compare the two ARIs.

In [12]:
six = ["card_arrival", "lost_or_stolen_card", "exchange_rate", "top_up_failed", "cash_withdrawal_charge", "passcode_forgotten"]

# 1. Messages and vectors of the six intents
mask = train["category"].isin(six).to_numpy()
subset = train[mask]
subset_vectors = train_vectors[mask]
print(subset.shape, subset_vectors.shape)

# 2. KMeans and crosstab
kmeans = KMeans(n_clusters=6, n_init=10, random_state=0).fit(subset_vectors)
display(pd.crosstab(kmeans.labels_, subset["category"], rownames=["cluster"]))

# 3. ARI
ari_embed = adjusted_rand_score(subset["category"], kmeans.labels_)

# 4. The same with TF-IDF
kmeans_tfidf = KMeans(n_clusters=6, n_init=10, random_state=0).fit(train_tfidf[mask])
display(pd.crosstab(kmeans_tfidf.labels_, subset["category"], rownames=["cluster"]))
ari_tfidf = adjusted_rand_score(subset["category"], kmeans_tfidf.labels_)

print(f"ARI, embeddings: {ari_embed:.2f}")
print(f"ARI, TF-IDF:     {ari_tfidf:.2f}")

(774, 2) (774, 384)


category,card_arrival,cash_withdrawal_charge,exchange_rate,lost_or_stolen_card,passcode_forgotten,top_up_failed
cluster,,,,,,
0,0,0,0,0,0,129
1,0,177,1,0,0,2
2,1,0,0,0,105,2
3,105,0,0,79,0,12
4,0,0,111,0,0,0
5,47,0,0,3,0,0


category,card_arrival,cash_withdrawal_charge,exchange_rate,lost_or_stolen_card,passcode_forgotten,top_up_failed
cluster,,,,,,
0,0,0,104,0,0,0
1,0,0,0,0,0,132
2,0,168,0,0,0,0
3,42,0,0,1,0,0
4,0,0,0,0,62,0
5,111,9,8,81,43,13


ARI, embeddings: 0.79
ARI, TF-IDF:     0.62


**Checkpoint:** an ARI of about 0.8 for the embeddings and about 0.6 for TF-IDF.

**Question:** which intents does KMeans mix or split, and why might that happen? Would the clusters be useful to a bank that has no intents yet?

With embeddings, four intents get their own almost pure cluster (`top_up_failed`, `cash_withdrawal_charge`, `passcode_forgotten`, `exchange_rate`). `card_arrival` and `lost_or_stolen_card` are mixed in one cluster (105 + 79) and `card_arrival` is also split across a second cluster (47). Since KMeans must make exactly 6 groups, splitting one intent forces it to merge two others. They mix because both are about a customer who does not have their card: *Where is my card?* fits both, and only the reason (still in the post or lost) differs. The split probably reflects two ways of asking about card arrival. TF-IDF has the same card mix, but also a catch-all cluster with leftovers from every intent, and it splits `passcode_forgotten` 62/43.

The clusters would be useful to a bank with no intents yet, as a starting point. Most clusters correspond to one clear topic, so a person can read a few messages from each, name it, and split the mixed card cluster by hand. That is much faster than labelling thousands of messages from scratch. In practice the bank would not know the right number of clusters either, so it would compare several values of k with the silhouette score.

## Extra task: know when you don't know

A search tool should not pretend. When even the best match is not very similar, it is better to say *I'm not sure* and pass the message to a person.

1. For each test message, take the score of its best match (`similarity.max(axis=1)`).
2. For the thresholds 0.5, 0.6, 0.7, 0.8 and 0.9, compute (a) the share of test messages whose best score is at least the threshold (the messages the tool would answer) and (b) the top-1 accuracy on those messages only. Present the results as a table.
3. Which threshold would you recommend to the support team, and why?

Bonus: instead of taking the intent of the single best match, take the most common intent among the top 3, 5 and 10 matches (a majority vote, like the k-nearest neighbours classifier). Does it improve the accuracy? Then try a few `TfidfVectorizer` settings from Day 1 (`stop_words="english"`, `ngram_range=(1, 2)`): can any of them close the gap with the embeddings?

In [13]:
# Extra task

*Your conclusions here.*